In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain.tools import tool
from typing import Dict, Any
from tavily import TavilyClient
from langchain_community.utilities import SQLDatabase

tavily_client = TavilyClient()

db = SQLDatabase.from_uri("sqlite:///resources/Chinook.db")


@tool
def web_search(query: str) -> Dict[str, Any]:

    """Search the web for information"""

    return tavily_client.search(query)

@tool
def sql_query(query: str) -> str:

    """Obtain information from the database using SQL queries"""

    try:
        return db.run(query)
    except Exception as e:
        return f"Error: {e}"

In [3]:
from dataclasses import dataclass

@dataclass
class UserRole:
    user_role: str = "external"

In [4]:
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from typing import Callable

@wrap_model_call
def dynamic_tool_call(request: ModelRequest, 
handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:

    """Dynamically call tools based on the runtime context"""

    user_role = request.runtime.context.user_role
    
    if user_role == "internal":
        pass # internal users get access to all tools
    else:
        tools = [web_search] # external users only get access to web search
        request = request.override(tools=tools) 

    return handler(request)

In [5]:
from langchain.agents import create_agent

agent = create_agent(
    model="gpt-5-nano",
    tools=[web_search, sql_query],
    middleware=[dynamic_tool_call],
    context_schema=UserRole
)

In [6]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="How many artists are in the database?")]},
    context={"user_role": "external"}
)

print(response["messages"][-1].content)

I don’t have access to your database, so I can’t see the count directly. Which database or system are you using, and what is the table/collection name?

If you share the schema or the exact table, I can give you the precise query. Here are common options you can run:

- SQL databases (MySQL, PostgreSQL, SQLite)
  - Basic count:
    - SELECT COUNT(*) AS artist_count FROM artists;
  - Exclude soft-deleted rows (if you have a deleted_at column or a status flag):
    - SELECT COUNT(*) AS artist_count FROM artists WHERE deleted_at IS NULL;
    - or: SELECT COUNT(*) AS artist_count FROM artists WHERE status = 'active';

- MongoDB
  - Basic count:
    - db.artists.countDocuments({});
  - If you want an estimated count:
    - db.artists.estimatedDocumentCount();

- If you want distinct artists by name:
  - SQL: SELECT COUNT(DISTINCT name) FROM artists;
  - MongoDB: db.artists.distinct("name").length (in the shell) or an aggregation.

If you can share your database type, the table/collection na